<h1>Radial velocity method: star <i>and</i> planet</h1>

A planet does not orbit a fixed star, but rather both the star and planet orbit their common **centre of mass**. Their orbits are ellipses of the same shape and eccentricity, but opposite orientation (i.e. they are always on opposite sides of the centre of mass). The sizes of the two orbits, and the velocities of the two bodies, are inversely proportional to their masses. The radial velocity method measures the star's small reflex motion. The planet's velocity, which is much larger than the star's, is not measured because planets are typically too close and too faint compared to their host stars for us to measure their spectral lines (and the shift that indicates their velocity). (A few _very_ hot planets close to their stars, e.g. $\tau$ Boötis b, have had their own orbital velocity measured with high-resolution infrared spectroscopy but this is not possible for the planets we will consider.)

The figure below shows both bodies at once:
- **top left:** the whole system, seen from above (birds-eye view looking down on the system's orbital plane), with the centre of mass marked by a cross;
- **top right:** the star's small orbit, in solar radii, with the star drawn to scale (shaded disk); the title gives the size of the star's orbit;
- **bottom left:** the star's radial velocity, which is what we measure from the shifts in its spectral lines via the Doppler effect;
- **bottom right:** the companion's radial velocity, which is normally _not_ measured (unless the companion is itself a star, bright enough to be seen).

Conventions (as in the radial velocity notebook): the observer is far away, in the direction of the negative $x$-axis; the arrow in the top-left panel shows this direction projected onto the orbital plane. A positive radial velocity means moving away from the observer (redshifted spectral lines), while a negative radial velocity means moving towards the observer (blueshifted spectral lines). The inclination $i$ is the angle between the system's orbital plane and the plane of sky (this is the same as the angle between our line of sight and the perpendicular to the orbital plane): $i=90^\circ$ means we see the orbit edge-on, $i=0^\circ$ face-on. We only measure the part of the velocity along our line of sight, $v_{\rm rad} = v_x \sin i$. The separation of the two bodies (the semi-major axis of their relative orbit) is fixed at 1 AU, so the orbital period changes when you change the masses (Kepler's third law).

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import Circle
from IPython.display import display, HTML
import ipywidgets as widgets

# Figures and animations controlled by sliders are shown in their own output area (widgets.Output).
# show_in() replaces what an output area shows, sending the new content to the browser once.
# (Using 'with out: display(...)' instead sends it back and forth, several MB per animation, and
# at the top level of a cell it can make Voila, the app view, hang.)
def show_in(out, obj):
    with out.hold_sync():
        out.outputs = ()
        out.append_display_data(obj)

from nbody import kepler as kp

# Try and make plots look nice
plt.rcParams["font.family"] = 'serif'
plt.rcParams["mathtext.fontset"] = 'stix'
plt.rcParams["font.serif"] = ['Palatino', 'TeX Gyre Pagella', 'DejaVu Serif']   # first available is used
plt.rcParams['pdf.fonttype'] = 42

<h2>Star and planet around their centre of mass</h2>

Use the sliders to change the system:
- **Orbital phase**: moves both bodies along their orbits (0 = pericentre, 0.5 = half an orbit later). For a circular orbit ($e=0$) there is no pericentre: phase 0 is then simply the moment when the planet is in the direction $\omega$.
- **Companion mass**, from 0.1 Jupiter masses up to about a solar mass (above about 13 Jupiter masses the companion is a brown dwarf, and above about 80 Jupiter masses it is a star). If you make the companion heavier than the star, the two swap roles: the "companion" is then the brighter star.
- **Star mass**, **eccentricity**, and **inclination** $i$.
- **Pericentre direction** $\omega$: the direction in which the planet is closest to the star, measured anticlockwise from the $+x$ axis. (The radial velocity notebook calls this the longitude of pericentre and uses radians. Exoplanet catalogues quote an "argument of periastron", which is measured from a different reference direction, so its values cannot be compared directly with this slider.)

The **Animate one orbit** button makes an animation of a full orbit for the current settings (this takes a few seconds).

In [ ]:
a = 1.0   # separation of star and companion: semi-major axis of the relative orbit (AU)


def star_radius(m_star):
    # Rough radius of a main-sequence star (solar radii)
    return m_star**0.8


def companion_kind(m_jup):
    if m_jup < 13:
        return 'planet'
    if m_jup < 80:
        return 'brown dwarf'
    return 'companion star'


def num(x):
    # Number with 3 significant digits, without powers of ten (e.g. 1000, 28.4, 0.000955)
    return np.format_float_positional(x, precision=3, fractional=False, trim='-')


def make_figure(dpi=None):
    fig, axs = plt.subplots(2, 2, figsize=(10, 8.4), dpi=dpi)
    plt.close(fig)          # show it only when we ask for it
    return fig, axs.ravel()


def draw(axs, m_comp_jup, m_star, e, omega_deg, inc_deg):
    # Draw the orbits and radial-velocity curves; return a function that puts the bodies
    # (and the markers on the velocity curves) at a given orbital phase.
    ax_sys, ax_star, ax_rv_star, ax_rv_comp = axs
    for ax in axs:
        ax.clear()

    m_comp = m_comp_jup*kp.M_JUP                   # companion mass in solar masses
    omega, inc = np.radians(omega_deg), np.radians(inc_deg)
    kind = companion_kind(m_comp_jup)
    orbit = kp.Orbit(kp.plot_times(kp.period(a, m_star + m_comp), e), m_star, m_comp, a, e, omega)
    P = orbit.period
    K_star, K_comp = kp.semi_amplitude(m_star, m_comp, a, e, inc)
    # Show the star's velocity in m/s, unless it is large
    star_unit, star_fac = ('m/s', 1000) if K_star < 1 else ('km/s', 1)

    # Whole system, seen from above
    L = 1.15*max(np.hypot(orbit.comp_x, orbit.comp_y).max(), np.hypot(orbit.star_x, orbit.star_y).max())
    ax_sys.plot(orbit.comp_x, orbit.comp_y, color='green', lw=1, alpha=0.6)
    ax_sys.plot(orbit.star_x, orbit.star_y, color='orange', lw=1)
    ax_sys.plot(0, 0, '+', color='k', ms=12, mew=1.5)
    comp_dot, = ax_sys.plot([], [], 'o', color='green', ms=10, label=kind, zorder=6)
    star_dot, = ax_sys.plot([], [], marker=r'$\star$', color='orange', ms=18, ls='None', label='star', zorder=6)
    ax_sys.annotate('', xy=(-0.95*L, -0.88*L), xytext=(-0.45*L, -0.88*L), arrowprops=dict(arrowstyle='->'))
    ax_sys.text(-0.95*L, -0.82*L, 'towards observer (projected)')
    ax_sys.set_xlim(-L, L)
    ax_sys.set_ylim(-L, L)
    ax_sys.set_aspect('equal')
    ax_sys.set_xlabel(r'$x$ (AU)')
    ax_sys.set_ylabel(r'$y$ (AU)')
    ax_sys.legend(loc='upper right', fontsize=9)
    ax_sys.set_title(f'Star and {kind}, seen from above')

    # The star's orbit in solar radii, with the star drawn to scale
    sx, sy = orbit.star_x/kp.R_SUN, orbit.star_y/kp.R_SUN
    R = star_radius(m_star)
    Ls = 1.2*max(np.hypot(sx, sy).max(), R)
    a_star = a*m_comp/(m_star + m_comp)            # semi-major axis of the star's own orbit (AU)
    ax_star.plot(sx, sy, color='orange', lw=2)
    ax_star.plot(0, 0, '+', color='k', ms=12, mew=1.5)
    star_disk = ax_star.add_patch(Circle((0, 0), R, color='orange', alpha=0.25, lw=0))
    star_centre, = ax_star.plot([], [], 'o', color='darkorange', ms=4)
    ax_star.set_xlim(-Ls, Ls)
    ax_star.set_ylim(-Ls, Ls)
    ax_star.set_aspect('equal')
    ax_star.set_xlabel(r'$x$ (solar radii)')
    ax_star.set_ylabel(r'$y$ (solar radii)')
    ax_star.set_title(f"Star's orbit: $a_*$ = {num(a_star/kp.R_SUN)} $R_\\odot$ = {num(a_star)} AU")

    # Radial velocities over one period (a little more, so that the fast pericentre passage
    # at the start and end of the orbit is not cut off at the edges of the plot)
    t = orbit.t
    t = np.concatenate([t[t > 0.95*P] - P, t, t[(t > 0) & (t < 0.05*P)] + P])
    rv_star, rv_comp = kp.Orbit(t, m_star, m_comp, a, e, omega).radial_velocity(inc)     # km/s
    days = t*kp.DAYS_PER_YEAR

    # Radial velocity of the star: measurable
    ax_rv_star.plot(days, star_fac*rv_star, color='orange')
    star_rv_dot, = ax_rv_star.plot([], [], 'o', color='orange', ms=8)
    ax_rv_star.set_ylabel(r'$v_{\rm rad}$ (' + star_unit + ')')
    star_note = 'measured' if m_comp <= m_star else 'the fainter star'
    ax_rv_star.set_title(f'Star: radial velocity ({star_note}), $K_*$ = {num(star_fac*K_star)} {star_unit}')

    # Radial velocity of the companion: normally not measurable for a planet
    ax_rv_comp.plot(days, rv_comp, color='green', ls='--', alpha=0.6)
    comp_rv_dot, = ax_rv_comp.plot([], [], 'o', color='green', ms=8, alpha=0.6)
    ax_rv_comp.set_ylabel(r'$v_{\rm rad}$ (km/s)')
    if kind != 'companion star':
        seen = 'normally not measurable: far too faint'
    elif m_comp > m_star:
        seen = 'heavier than the "star", so in fact the brighter one'
    else:
        seen = 'measurable if bright enough'
    ax_rv_comp.set_title(f'{kind.capitalize()}: $K_{{\\rm p}}$ = {num(K_comp)} km/s\n({seen})')

    for ax in (ax_rv_star, ax_rv_comp):
        ax.axhline(0, color='grey', lw=0.5)
        ax.set_xlim(-0.03*P*kp.DAYS_PER_YEAR, 1.03*P*kp.DAYS_PER_YEAR)
        ax.set_xlabel('time since pericentre (days)')
        if K_star == 0:           # seen face-on: nothing moves along our line of sight
            ax.set_ylim(-1, 1)
            ax.text(0.5, 0.6, 'seen face-on: no motion along our line of sight',
                    transform=ax.transAxes, ha='center')

    fig = ax_sys.figure
    fig.suptitle(f'Period $P$ = {P*kp.DAYS_PER_YEAR:.0f} days;   {kind} mass $m$ = {num(m_comp_jup)} $M_{{\\rm J}}$'
                 f' = {num(m_comp)} $M_\\odot$;   $m \\sin i$ = {num(m_comp_jup*np.sin(inc))} $M_{{\\rm J}}$')
    fig.tight_layout()

    def move(phase):
        now = kp.Orbit(phase*P, m_star, m_comp, a, e, omega)
        v_star, v_comp = now.radial_velocity(inc)
        comp_dot.set_data([now.comp_x], [now.comp_y])
        star_dot.set_data([now.star_x], [now.star_y])
        star_centre.set_data([now.star_x/kp.R_SUN], [now.star_y/kp.R_SUN])
        star_disk.center = (now.star_x/kp.R_SUN, now.star_y/kp.R_SUN)
        star_rv_dot.set_data([phase*P*kp.DAYS_PER_YEAR], [star_fac*v_star])
        comp_rv_dot.set_data([phase*P*kp.DAYS_PER_YEAR], [v_comp])
        return comp_dot, star_dot, star_centre, star_disk, star_rv_dot, comp_rv_dot

    return move

In [ ]:
style = {'description_width': 'initial'}
w_phase = widgets.FloatSlider(value=0, min=0, max=1, step=0.01, description='Orbital phase (0 = pericentre):',
                              continuous_update=False, readout_format='.2f', style=style)
w_mass = widgets.FloatLogSlider(value=1, base=10, min=-1, max=3, step=0.02,
                                description='Companion mass (Jupiter masses):',
                                continuous_update=False, readout_format='.3r', style=style)
w_star = widgets.FloatSlider(value=1.0, min=0.1, max=2.0, step=0.05, description='Star mass (solar masses):',
                             continuous_update=False, readout_format='.2f', style=style)
w_ecc = widgets.FloatSlider(value=0.0, min=0, max=0.95, step=0.01, description='Eccentricity:',
                            continuous_update=False, readout_format='.2f', style=style)
w_omega = widgets.FloatSlider(value=0, min=0, max=360, step=5, description='Pericentre direction ω (degrees):',
                              continuous_update=False, readout_format='.0f', style=style)
w_inc = widgets.FloatSlider(value=90, min=0, max=90, step=1, description='Inclination i (degrees, 90 = edge-on):',
                            continuous_update=False, readout_format='.0f', style=style)
w_animate = widgets.Button(description='Animate one orbit')

out_fig = widgets.Output()
out_anim = widgets.Output()
display(w_phase, w_mass, w_star, w_ecc, w_omega, w_inc, out_fig, w_animate, out_anim)

fig, axs = make_figure()


def current_settings():
    return dict(m_comp_jup=w_mass.value, m_star=w_star.value, e=w_ecc.value,
                omega_deg=w_omega.value, inc_deg=w_inc.value)


# When any slider is moved, redraw the figure
def update(change=None):
    if change is not None and change['owner'] is not w_phase:
        out_anim.outputs = ()           # an existing animation no longer matches the settings
    with out_fig:                       # any error message appears in this output area
        move = draw(axs, **current_settings())
        move(w_phase.value)
        show_in(out_fig, fig)


# When the button is pressed, make an animation of one orbit (smaller figure, 60 frames)
def make_animation(button):
    with out_anim:                      # any error message appears in this output area
        show_in(out_anim, HTML('<p><i>Making the animation, please wait a few seconds...</i></p>'))
        fig_anim, axs_anim = make_figure(dpi=60)
        move = draw(axs_anim, **current_settings())
        phases = np.linspace(0, 1, 60, endpoint=False)
        anim = animation.FuncAnimation(fig_anim, lambda k: move(phases[k]), frames=len(phases), interval=100)
        show_in(out_anim, HTML(anim.to_jshtml(default_mode='loop')))


for w in (w_phase, w_mass, w_star, w_ecc, w_omega, w_inc):
    w.observe(update, names='value')
w_animate.on_click(make_animation)
update()